# 5 — EEG Preprocessing with EEGPrep: Familiar vs Unfamiliar Faces

**Colab port of the EEGDash example**
[`project_face_familiarity`](https://eegdash.org/generated/auto_examples/applied/project_face_familiarity.html)

One participant of the **Wakeman & Henson face-recognition study** (OpenNeuro `ds002718`).
The task: decide from a single trial of EEG whether the face on screen was **famous** or
**unfamiliar**. Two classes, chance is 0.5.

The point of this notebook is the *preprocessing*, not the network. `EEGPrep` is
braindecode's one-call cleaning pipeline — resampling, high-pass filtering, bad-channel
handling and ASR burst removal — applied to the continuous recording before any epoching
happens. The classifier at the end (`ShallowFBCSPNet`) is deliberately small, so that what
you see in the accuracy is mostly the quality of the signal you fed it.

### Route

1. Pull one subject from EEGDash (streams from OpenNeuro, cached locally)
2. Clean the continuous signal with `EEGPrep`
3. Epoch on face events and split trials into train/test
4. Train `ShallowFBCSPNet` and read the learning curve

**Runtime:** CPU is fine here (one subject, a small convnet). A GPU makes step 4 quicker.
Total ~5–10 min, most of it the download.

## 0 · Install

EEGDash pulls the BIDS dataset; braindecode supplies `EEGPrep` and the model. The runtime
restart afterwards is expected and necessary — it lets the freshly installed packages load
cleanly.

In [ ]:
%pip install -q eegdash braindecode

import IPython
print("Install finished — restarting the runtime.")
print("This 'crash' notice is expected. Continue at Section 1 below.")
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.model_selection import train_test_split

from eegdash import EEGDashDataset
from braindecode import EEGClassifier
from braindecode.models import ShallowFBCSPNet
from braindecode.preprocessing import (
    EEGPrep,
    Preprocessor,
    create_windows_from_events,
    preprocess,
)
from braindecode.util import set_random_seeds

import mne
mne.set_log_level("ERROR")

# SFREQ is the target sampling rate everything downstream assumes
SFREQ = 128

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} · device {DEVICE}")

## 1 · Load one participant of ds002718

`EEGDashDataset` queries the EEGDash index and fetches only the files matching your
filters — here a single subject and a single task, rather than the whole 19-subject
dataset. Files land in `cache_dir`, so re-running the cell is free.

In Colab the cache sits on the ephemeral disk and disappears when the runtime recycles. If
you want it to survive, mount Drive and point `EEGDASH_CACHE_DIR` at a folder there.

In [ ]:
# Optional: keep the download across sessions by caching into Google Drive.
# from google.colab import drive
# drive.mount("/content/drive")
# os.environ["EEGDASH_CACHE_DIR"] = "/content/drive/MyDrive/eegdash_cache"

CACHE_DIR = Path(os.environ.get("EEGDASH_CACHE_DIR", "~/.eegdash_cache")).expanduser()

ds = EEGDashDataset(
    cache_dir=CACHE_DIR,
    dataset="ds002718",
    subject="018",
    task="FaceRecognition",
)
print(ds.description)

## 2 · Clean the continuous signal with EEGPrep

Three preprocessors, applied in order to the continuous recording:

- **`pick`** — keep EEG channels only; this recording also carries EOG and stimulus
  channels that the network has no business seeing.
- **`lambda x: x * 1e6`** — MNE stores volts, and a network initialised for order-1 inputs
  will not train on values around `1e-5`. Converting to microvolts is not cosmetic; skip it
  and the model sits at chance.
- **`EEGPrep`** — the actual cleaning: resample to 128 Hz, high-pass with a transition band
  from 0.25 Hz (full stop) to 0.75 Hz (full pass), then ASR burst removal rejecting
  components beyond 10 SD of the calibration data.

The argument worth pausing on is `bad_window_max_bad_channels=None`. It switches off
bad-*window* removal. Windows are dropped on the time axis, so removing them would delete
trials and desynchronise the event structure the next step depends on. Leaving the time
axis intact means every trial survives into epoching.

This cell does the heavy lifting — give it a couple of minutes.

In [ ]:
preprocess(
    ds,
    [
        Preprocessor("pick", picks="eeg"),
        Preprocessor(
            lambda x: x * 1e6
        ),  # V -> uV: the network does not train on V-scale inputs
        EEGPrep(
            resample_to=SFREQ,
            highpass_frequencies=(
                0.25,
                0.75,
            ),  # transition band: full stop at 0.25 Hz, passband from 0.75 Hz
            burst_removal_cutoff=10.0,  # ASR: reject components beyond 10 SD of the calibration data
            bad_window_max_bad_channels=None,  # no bad-window removal: time axis intact, every trial survives
        ),
    ],
)
print("EEGPrep done.")

## 3 · Epoch familiar vs unfamiliar faces and split the trials

The BIDS events distinguish six conditions: each face type appears as a *new* presentation
and as an *early* or *late* repetition. Familiarity is what we are decoding, so the
`mapping` collapses all three famous conditions to class 0 and all three unfamiliar ones to
class 1. Scrambled faces are absent from the mapping and therefore dropped.

Each trial runs from 200 ms before the face to 800 ms after — the window holding the N170
and the later familiarity effects.

The split is a plain stratified random split over trials. Note what that does and does not
buy you: it keeps the class balance honest, but train and test trials come from the same
recording session of the same participant, so the accuracy below is a *within-subject*
number. It says nothing about a new participant.

In [ ]:
mapping = {
    "famous_new": 0,
    "famous_second_early": 0,
    "famous_second_late": 0,
    "unfamiliar_new": 1,
    "unfamiliar_second_early": 1,
    "unfamiliar_second_late": 1,
}
windows = create_windows_from_events(
    ds,
    trial_start_offset_samples=int(-0.2 * SFREQ),
    trial_stop_offset_samples=int(0.8 * SFREQ),
    mapping=mapping,
)
X = np.stack([x for x, _, _ in windows])
y = windows.get_metadata()["target"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=0
)
print(
    f"X {X.shape} | train {len(y_train)} | test {len(y_test)} | classes {np.bincount(y)}"
)

## 4 · Train ShallowFBCSPNet

`ShallowFBCSPNet` is the deep-learning restatement of filter-bank CSP: a temporal
convolution, a spatial convolution across channels, then square–pool–log. Small enough to
train on a few hundred trials without a GPU, which is exactly the regime one participant
puts you in.

`EEGClassifier` is braindecode's skorch wrapper — it gives the model an sklearn-style
`fit`/`score` interface and records a per-epoch history we plot next. It holds out an
internal validation split from the training trials by default, which is where
`valid_acc` in the learning curve comes from.

In [ ]:
set_random_seeds(seed=0, cuda=False)
clf = EEGClassifier(
    ShallowFBCSPNet(n_chans=X.shape[1], n_outputs=2, n_times=X.shape[2]),
    optimizer=torch.optim.AdamW,
)
clf.fit(X_train, y_train, epochs=30)

## 5 · Score the held-out trials and look at the learning curve

Two things to read off the plot. The **training loss** should fall steadily — if it is flat,
suspect the microvolt scaling. The **validation accuracy** against the dashed chance line
tells you whether anything generalises; on a single participant expect something modest and
noisy rather than a clean climb.

Single-trial familiarity decoding is genuinely hard. A test accuracy meaningfully above
0.5 here is a real result, not a disappointing one.

In [ ]:
print(f"test accuracy: {clf.score(X_test, y_test):.3f} (chance 0.5)")

# Learning curve from the skorch history: training loss and validation accuracy per epoch
history = clf.history
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(history[:, "epoch"], history[:, "train_loss"], marker="o", label="train loss")
ax.plot(
    history[:, "epoch"],
    history[:, "valid_acc"],
    marker="s",
    label="validation accuracy",
)
ax.axhline(0.5, ls="--", color="gray")  # chance level for the accuracy curve
ax.set(xlabel="epoch", ylim=(0, 1))
ax.legend()
plt.show()

## What to take away

- **`EEGPrep` is one call, but it is four decisions.** Resampling rate, filter transition
  band, ASR cutoff, and whether bad windows may be dropped. The last one interacts directly
  with epoching — `None` here because trials must survive.
- **Units matter more than architecture.** The `* 1e6` line is the difference between
  learning and not learning.
- **The split defines the claim.** A stratified trial split measures within-session
  decoding. For a claim about new participants you need subject-grouped folds — which is
  precisely what the companion fine-tuning notebook does.

### Things to try

- Drop the `EEGPrep` preprocessor and re-run: how much accuracy does the cleaning buy?
- Change `burst_removal_cutoff` to 5 (more aggressive) or 20 (barely any) and compare.
- Swap `ShallowFBCSPNet` for `EEGNetv4` or `Deep4Net` from `braindecode.models`.
- Load a second subject and test across participants instead of across trials.